
#### Requisitos para o arquivo torneios.csv

Objetivo: realizar um processo de ETL e disponibilizar os dados em uma camada apropriada

Ler -> Transformar -> Escrever (ETL)

1. Ler o arquivo csv do Data Lake
2. Renomear as colunas para português utilizando o **snake case**
3. Definir o esquema de dados correto
4. Precisa ter uma coluna com a data em que o arquivo foi consumido
5. Salvar os dados em formato **parquet** na camada adequada
6. Bônus: Criar uma coluna com a palavra 'Brasil'


#### 1. Lendo o arquivo CSV

In [0]:
%run "../Modulo 2/Configurando o storage"

In [0]:
%run "../Modulo 4/Variaveis"

In [0]:
%run "../Modulo 4/funcoes"

In [0]:

caminho_arquivo_partidas = f"{caminho_bronze}partidas"


In [0]:
display(dbutils.fs.ls(caminho_bronze))

In [0]:
from functools import reduce

opcoes = {"header": "true", "sep": ",", "inferSchema": "true"}

dfs = []
for f in dbutils.fs.ls(f"{caminho_bronze}partidas/"):
    if f.name.endswith(".csv"):
        df = spark.read.options(**opcoes).option("encoding", "UTF-8").csv(f.path)
        if df.columns[0] != "id":
            df = (spark.read.options(**opcoes)
                  .option("encoding", "UTF-16")
                  .option("multiLine", "true")
                  .csv(f.path))
            print(f.name, "-> UTF-16")
        else:
            print(f.name, "-> UTF-8")
        dfs.append(df)

partidas_df = reduce(lambda a, b: a.unionByName(b), dfs)

In [0]:

# caminho_arquivo_partidas_2023 = f"{caminho_bronze}partidas/partidas_2023.csv"

In [0]:
#partidas_df.printSchema()
#display(partidas_df.limit(5))

In [0]:
#display(dbutils.fs.ls(caminho_bronze))

In [0]:
#partidas_df_2023 = (spark.read
 #              .option("encoding", "UTF-8")
  #             .option("header", "true")
   #            .option("sep", ",")
    #           .option("inferSchema", "true")
     #          .csv(caminho_arquivo_partidas_2023))

In [0]:
#partidas_df = partidas_df.union(partidas_df_2023)

In [0]:
#partidas_df_2023.printSchema()
#display(partidas_df_2023.limit(5))

In [0]:
display(partidas_df)

Databricks visualization. Run in Databricks to view.

In [0]:
partidas_df.printSchema()

In [0]:
partidas_df_renomeado = partidas_df \
    .withColumnRenamed("id", "id_partida") \
    .withColumnRenamed("referee", "nm_juiz") \
    .withColumnRenamed("date", "dt_partida") \
    .withColumnRenamed("venue.id", "id_estadio") \
    .withColumnRenamed("status.long", "ds_status") \
    .withColumnRenamed("league.id", "id_torneio") \
    .withColumnRenamed("league.season", "ano_torneio") \
    .withColumnRenamed("league.round", "nr_partida") \
    .withColumnRenamed("teams.home.id", "id_time_casa") \
    .withColumnRenamed("teams.away.id", "id_time_fora") \
    .withColumnRenamed("goals.home", "qt_gols_casa") \
    .withColumnRenamed("goals.away", "qt_gols_fora") \
    .withColumnRenamed("score.halftime.home", "qt_gols_pr_tempo_casa") \
    .withColumnRenamed("score.halftime.away", "qt_gols_pr_tempo_fora") \
    .withColumnRenamed("score.fulltime.home", "qt_gols_jogo_casa") \
    .withColumnRenamed("score.fulltime.away", "qt_gols_jogo_fora")

In [0]:
partidas_df_renomeado.display()

In [0]:
from pyspark.sql.types import BooleanType, DateType, IntegerType
from pyspark.sql.functions import col

In [0]:
partidas_df_retipado = partidas_df_renomeado.withColumn(
    "dt_partida", partidas_df_renomeado.dt_partida.cast(DateType())
)

In [0]:
partidas_df_retipado.printSchema()

In [0]:
partidas_df_retipado = partidas_df_renomeado \
    .withColumn("dt_partida", partidas_df_renomeado.dt_partida.cast('date')) \
    .withColumn("qt_gols_casa", col('qt_gols_casa').cast(IntegerType())) \
    .withColumn("qt_gols_fora", col('qt_gols_fora').cast('integer')) \
    .withColumn("qt_gols_pr_tempo_casa", col('qt_gols_pr_tempo_casa').cast(IntegerType())) \
    .withColumn("qt_gols_pr_tempo_fora", col('qt_gols_pr_tempo_fora').cast(IntegerType())) \
    .withColumn("qt_gols_jogo_casa", col('qt_gols_jogo_casa').cast(IntegerType())) \
    .withColumn("qt_gols_jogo_fora", col("qt_gols_jogo_fora").cast("double").cast("int"))

In [0]:
partidas_df_retipado.printSchema()

In [0]:
partidas_df_retipado.display()

In [0]:
from pyspark.sql.functions import current_timestamp, lit

In [0]:
partidas_df_data = cria_coluna_data(partidas_df_renomeado)

In [0]:
display(partidas_df_data)

In [0]:
partidas_df_data.printSchema()

In [0]:
partidas_df_data.write.partitionBy('ano_torneio').parquet(f"{caminho_silver}partidas/", mode="overwrite")